# AstroCLIMB — Qwen3-VL-8B restricted-loss validation QLoRA

This experiment fine-tunes **Qwen3-VL-8B-Instruct** with cross-entropy restricted to the four valid label-token logits. It uses the reproducible 9,200/800 balanced split, trains for two epochs, evaluates and checkpoints every half epoch, and restores the checkpoint with the best validation macro-F1.

Only language `q_proj`, `k_proj`, `v_proj`, and `o_proj` modules receive rank-16 LoRA adapters; explicit module discovery and trainable-parameter checks prevent accidental vision adaptation. Select **GPU T4 x2** in Kaggle and attach the AstroCLIMB competition data.


In [1]:
# Preserve Kaggle's torch, torchvision, Pillow, and scikit-learn versions.
%pip install -q --upgrade --upgrade-strategy only-if-needed "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes==0.47.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
csv.field_size_limit(sys.maxsize)
print('torch:', torch.__version__)
print('CUDA has not been initialized:', not torch.cuda.is_initialized())


torch: 2.10.0+cu128
CUDA has not been initialized: True


In [3]:
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
TARGET_COLUMNS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
DIGIT_TO_LABEL = dict(enumerate(TARGET_COLUMNS))
VAL_PER_CLASS = 200
EXPECTED_TRAIN_ROWS = 9200
EXPECTED_VALIDATION_ROWS = 800
EXPECTED_TEST_ROWS = 10000
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
NUM_EPOCHS = 2
GRADIENT_ACCUMULATION = 8  # global batch = 1 x 2 GPUs x 8 = 16
REBUILD_CACHE = False
RUN_TEST_INFERENCE = False  # Enable only after this run is selected on validation.
TEST_LIMIT = None  # Keep None for the complete 10,000-row submission.
USE_SWAP_TTA = False
APPLY_MODALITY_MASK = False  # Keep False for a clean loss-only comparison.

WORK_ROOT = Path('/kaggle/working/astroclimb_qwen3vl8b_restricted_validation') if Path('/kaggle/working').exists() else Path('./astroclimb_qwen3vl8b_restricted_validation')
IMAGE_ROOT = WORK_ROOT / 'images'
TRAIN_MANIFEST = WORK_ROOT / 'train_9200.jsonl'
VALIDATION_MANIFEST = WORK_ROOT / 'validation_800.jsonl'
TEST_MANIFEST = WORK_ROOT / 'test_10000.jsonl'
ADAPTER_DIR = WORK_ROOT / 'best_adapter'
PREDICTION_DIR = WORK_ROOT / 'prediction_shards'
SUBMISSION_PATH = WORK_ROOT / 'submission.csv'
for path in [WORK_ROOT, IMAGE_ROOT, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def locate_csv(filename):
    for candidate in [
        Path('/kaggle/input/competitions/astroclimb') / filename,
        Path('/kaggle/input/astroclimb') / filename,
    ]:
        if candidate.exists():
            return candidate
    roots = [Path('/kaggle/input'), Path('data')]
    candidates = [p for root in roots if root.exists() for p in root.rglob(filename)]
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f'{filename} not found. Attach the AstroCLIMB competition data.')
    return candidates[0]

def locate_model():
    if Path('/kaggle/input').exists():
        configs = list(Path('/kaggle/input').rglob('config.json'))
        candidates = [p.parent for p in configs if 'qwen3' in str(p).lower() and 'vl' in str(p).lower() and '8b' in str(p).lower()]
        if candidates:
            return str(sorted(candidates, key=lambda p: len(str(p)))[0])
    return MODEL_ID

TRAIN_CSV = locate_csv('train.csv')
TEST_CSV = locate_csv('test.csv')
MODEL_PATH = locate_model()
print('Train:', TRAIN_CSV)
print('Test:', TEST_CSV)
print('Model:', MODEL_PATH)
print('Working directory:', WORK_ROOT)


Train: /kaggle/input/competitions/astroclimb/train.csv
Test: /kaggle/input/competitions/astroclimb/test.csv
Model: Qwen/Qwen3-VL-8B-Instruct
Working directory: /kaggle/working/astroclimb_qwen3vl8b_restricted_validation


## Select a permanent balanced validation split

The CSV is streamed once to select 200 validation IDs per class with reservoir sampling. This avoids loading multi-gigabyte base64 columns into memory and leaves exactly 9,200 rows for training.


In [4]:
def get_label(row):
    values = [int(float(row[column])) for column in TARGET_COLUMNS]
    if sum(values) != 1:
        raise ValueError(f'Invalid one-hot label for id={row.get("id")}: {values}')
    return values.index(1)

def select_validation_ids(path, per_class=200, seed=42):
    rng = random.Random(seed)
    reservoirs = {label: [] for label in range(4)}
    seen = {label: 0 for label in range(4)}
    started = time.perf_counter()
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        required = {'id', 'obj_1', 'obj_2', *TARGET_COLUMNS}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Missing train columns: {sorted(missing)}')
        for row_number, row in enumerate(reader, start=1):
            label = get_label(row)
            seen[label] += 1
            bucket = reservoirs[label]
            if len(bucket) < per_class:
                bucket.append(row['id'])
            else:
                position = rng.randrange(seen[label])
                if position < per_class:
                    bucket[position] = row['id']
            if row_number % 1000 == 0:
                print(f'Split scan {row_number} rows | {(time.perf_counter()-started)/60:.2f} min')
    selected = {identifier for ids in reservoirs.values() for identifier in ids}
    print('Rows seen:', {DIGIT_TO_LABEL[k]: v for k, v in seen.items()})
    print('Validation IDs:', len(selected))
    assert len(selected) == EXPECTED_VALIDATION_ROWS
    return selected

validation_ids = select_validation_ids(TRAIN_CSV, VAL_PER_CLASS, SEED)


Split scan 1000 rows | 0.45 min
Split scan 2000 rows | 1.49 min
Split scan 3000 rows | 2.08 min
Split scan 4000 rows | 2.08 min
Split scan 5000 rows | 3.00 min
Split scan 6000 rows | 3.51 min
Split scan 7000 rows | 3.51 min
Split scan 8000 rows | 4.45 min
Split scan 9000 rows | 4.96 min
Split scan 10000 rows | 4.96 min
Rows seen: {'same_figure': 1000, 'same_paper': 3000, 'related_papers': 3000, 'unrelated_papers': 3000}
Validation IDs: 800


## Decode and cache train, validation, and test objects

Images are decoded once, resized while preserving aspect ratio, and cached by SHA-256. Manifests contain local paths instead of base64 payloads. Existing complete manifests are reused unless `REBUILD_CACHE=True`.


In [5]:
def looks_like_image(value):
    if not isinstance(value, str):
        return False
    return value.lstrip().startswith(('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD', 'data:image'))

def decode_image(value):
    value = value.strip()
    if value.startswith('data:image'):
        value = value.split(',', 1)[1]
    image = Image.open(io.BytesIO(base64.b64decode(value, validate=False)))
    image.load()
    return image.convert('RGB')

def resize_to_area(image, max_pixels=MAX_PIXELS):
    width, height = image.size
    if width * height <= max_pixels:
        return image
    scale = math.sqrt(max_pixels / (width * height))
    return image.resize((max(1, round(width * scale)), max(1, round(height * scale))), Image.Resampling.LANCZOS)

def cache_object(value):
    if not looks_like_image(value):
        return {'kind': 'caption', 'value': value}
    digest = hashlib.sha256(value.encode('utf-8')).hexdigest()
    path = IMAGE_ROOT / f'{digest}.png'
    if not path.exists():
        image = resize_to_area(decode_image(value))
        image.save(path, format='PNG', compress_level=3)
    return {'kind': 'image', 'value': str(path)}

def count_lines(path):
    if not path.exists():
        return -1
    with path.open('r', encoding='utf-8') as handle:
        return sum(1 for _ in handle)

def manifests_are_complete():
    return (
        count_lines(TRAIN_MANIFEST) == EXPECTED_TRAIN_ROWS
        and count_lines(VALIDATION_MANIFEST) == EXPECTED_VALIDATION_ROWS
        and count_lines(TEST_MANIFEST) == EXPECTED_TEST_ROWS
    )

def build_manifests():
    counts = {'train': 0, 'validation': 0, 'test': 0}
    class_counts = {'train': [0] * 4, 'validation': [0] * 4}
    modality_counts = {'train': {}, 'validation': {}, 'test': {}}
    started = time.perf_counter()
    with (
        TRAIN_CSV.open('r', encoding='utf-8', newline='') as source,
        TRAIN_MANIFEST.open('w', encoding='utf-8') as train_output,
        VALIDATION_MANIFEST.open('w', encoding='utf-8') as validation_output,
    ):
        reader = csv.DictReader(source)
        for index, row in enumerate(reader, start=1):
            label = get_label(row)
            obj_1, obj_2 = cache_object(row['obj_1']), cache_object(row['obj_2'])
            modality = obj_1['kind'][0].upper() + obj_2['kind'][0].upper()
            split = 'validation' if row['id'] in validation_ids else 'train'
            record = {'id': row['id'], 'obj_1': obj_1, 'obj_2': obj_2, 'label': label, 'modality': modality}
            destination = validation_output if split == 'validation' else train_output
            destination.write(json.dumps(record, ensure_ascii=False) + '\n')
            counts[split] += 1
            class_counts[split][label] += 1
            modality_counts[split][modality] = modality_counts[split].get(modality, 0) + 1
            if index % 250 == 0:
                print(f'Train preprocessing {index}/10000 | {(time.perf_counter()-started)/60:.2f} min')
    with TEST_CSV.open('r', encoding='utf-8', newline='') as source, TEST_MANIFEST.open('w', encoding='utf-8') as output:
        reader = csv.DictReader(source)
        required = {'id', 'obj_1', 'obj_2'}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Missing test columns: {sorted(missing)}')
        for index, row in enumerate(reader, start=1):
            obj_1, obj_2 = cache_object(row['obj_1']), cache_object(row['obj_2'])
            modality = obj_1['kind'][0].upper() + obj_2['kind'][0].upper()
            output.write(json.dumps({'id': row['id'], 'obj_1': obj_1, 'obj_2': obj_2, 'modality': modality}, ensure_ascii=False) + '\n')
            counts['test'] += 1
            modality_counts['test'][modality] = modality_counts['test'].get(modality, 0) + 1
            if index % 250 == 0:
                print(f'Test preprocessing {index}/10000 | {(time.perf_counter()-started)/60:.2f} min')
    print('Rows:', counts)
    print('Class counts:', class_counts)
    print('Modality counts:', modality_counts)
    assert counts == {'train': EXPECTED_TRAIN_ROWS, 'validation': EXPECTED_VALIDATION_ROWS, 'test': EXPECTED_TEST_ROWS}
    assert class_counts['validation'] == [VAL_PER_CLASS] * 4
    print(f'Total preprocessing: {(time.perf_counter()-started)/60:.2f} min')

if REBUILD_CACHE or not manifests_are_complete():
    build_manifests()
else:
    print('Reusing complete cached manifests.')
print('Manifest rows:', count_lines(TRAIN_MANIFEST), count_lines(VALIDATION_MANIFEST), count_lines(TEST_MANIFEST))
print('Cached PNGs:', len(list(IMAGE_ROOT.glob('*.png'))))
gc.collect()


Train preprocessing 250/10000 | 0.43 min
Train preprocessing 500/10000 | 0.85 min
Train preprocessing 750/10000 | 1.30 min
Train preprocessing 1000/10000 | 1.70 min
Train preprocessing 1250/10000 | 2.57 min
Train preprocessing 1500/10000 | 3.41 min
Train preprocessing 1750/10000 | 4.23 min
Train preprocessing 2000/10000 | 5.11 min
Train preprocessing 2250/10000 | 5.54 min
Train preprocessing 2500/10000 | 5.94 min
Train preprocessing 2750/10000 | 6.35 min
Train preprocessing 3000/10000 | 6.79 min
Train preprocessing 3250/10000 | 6.79 min
Train preprocessing 3500/10000 | 6.79 min
Train preprocessing 3750/10000 | 6.79 min
Train preprocessing 4000/10000 | 6.79 min
Train preprocessing 4250/10000 | 7.60 min
Train preprocessing 4500/10000 | 8.36 min
Train preprocessing 4750/10000 | 9.17 min
Train preprocessing 5000/10000 | 9.97 min
Train preprocessing 5250/10000 | 10.39 min
Train preprocessing 5500/10000 | 10.79 min
Train preprocessing 5750/10000 | 11.20 min
Train preprocessing 6000/10000 | 1

0

## Two-T4 restricted-loss QLoRA training

The worker calculates loss from only the four digit logits. With 9,200 rows, global batch 16, and two epochs, training performs 575 optimizer steps per epoch and 1,150 total. A milestone callback evaluates and checkpoints at steps 288, 575, 863, and 1,150—approximately 0.5, 1.0, 1.5, and 2.0 epochs. The adapter with the best validation macro-F1 is restored and exported.


### Visible restricted-loss training worker

This cell writes the complete worker as normal Python source. It is intentionally shown directly rather than hidden inside a base64 payload.


In [6]:
%%writefile train_restricted4_ddp.py
import argparse
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000

SYSTEM_PROMPT = """You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.""".strip()


def shorten_caption(text, max_chars=MAX_TEXT_CHARS):
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + "\n[...middle truncated...]\n" + text[-half:]


def object_content(number, obj):
    if obj["kind"] == "image":
        with Image.open(obj["value"]) as source:
            image = source.convert("RGB")
        return [
            {"type": "text", "text": f"Object {number} is a scientific figure:"},
            {"type": "image", "image": image},
        ]
    return [{"type": "text", "text": f"Object {number} is a figure caption:\n{shorten_caption(obj['value'])}"}]


def build_messages(row, swap=False):
    obj_1, obj_2 = row["obj_1"], row["obj_2"]
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({"type": "text", "text": "Classify their relationship. Reply with one digit only."})
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": content},
        {"role": "assistant", "content": [{"type": "text", "text": str(int(row["label"]))}]},
    ]


class ManifestDataset(torch.utils.data.Dataset):
    def __init__(self, path, random_swap=False):
        with Path(path).open("r", encoding="utf-8") as handle:
            self.rows = [json.loads(line) for line in handle]
        self.random_swap = random_swap

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = dict(self.rows[index])
        row["_swap"] = self.random_swap and random.random() < 0.5
        return row


class LabelOnlyCollator:
    def __init__(self, processor):
        self.processor = processor
        self.label_token_ids = []
        for digit in "0123":
            ids = processor.tokenizer.encode(digit, add_special_tokens=False)
            if len(ids) != 1:
                raise ValueError(f"Label {digit} is not a single token: {ids}")
            self.label_token_ids.append(ids[0])

    def __call__(self, features):
        if len(features) != 1:
            raise ValueError(f"Expected per-device batch 1, received {len(features)}")
        row = features[0]
        batch = self.processor.apply_chat_template(
            build_messages(row, swap=row.get("_swap", False)),
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt",
        )
        target_id = self.label_token_ids[int(row["label"])]
        positions = torch.where(batch["input_ids"][0] == target_id)[0]
        if not len(positions):
            raise RuntimeError("Assistant label token was not found in the rendered conversation.")
        labels = torch.full_like(batch["input_ids"], -100)
        labels[0, int(positions[-1])] = target_id
        batch["labels"] = labels
        return batch


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-manifest", required=True)
    parser.add_argument("--validation-manifest", required=True)
    parser.add_argument("--adapter-dir", required=True)
    parser.add_argument("--work-root", required=True)
    parser.add_argument("--model-path", required=True)
    parser.add_argument("--epochs", type=float, default=2.0)
    parser.add_argument("--gradient-accumulation", type=int, default=8)
    args = parser.parse_args()

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)

    # Import after rank device selection so optional CUDA probes use the correct T4.
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from sklearn.metrics import f1_score
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen3VLForConditionalGeneration,
        Trainer,
        TrainerCallback,
        TrainingArguments,
    )

    random.seed(SEED + local_rank)
    np.random.seed(SEED + local_rank)
    torch.manual_seed(SEED + local_rank)
    load_started = time.perf_counter()

    processor = AutoProcessor.from_pretrained(args.model_path, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    processor.tokenizer.padding_side = "right"
    collator = LabelOnlyCollator(processor)
    label_token_ids_cpu = torch.tensor(collator.label_token_ids, dtype=torch.long)
    if local_rank == 0:
        print(f"Label token IDs: {collator.label_token_ids}", flush=True)

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        args.model_path,
        quantization_config=quantization,
        dtype=torch.float16,
        attn_implementation="sdpa",
        device_map={"": local_rank},
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    target_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj"}
    language_targets = [
        name for name, _ in model.named_modules()
        if ".visual." not in f".{name}."
        and name.rsplit(".", 1)[-1] in target_suffixes
    ]
    if not language_targets:
        raise RuntimeError("No language LoRA targets were discovered.")
    model = get_peft_model(
        model,
        LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=language_targets,
        ),
    )
    trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    if any(".visual." in f".{name}." for name in trainable_names):
        raise RuntimeError("Language-only target selection unexpectedly made visual parameters trainable.")
    missing_suffixes = [
        suffix for suffix in target_suffixes
        if not any(f".{suffix}." in name for name in trainable_names)
    ]
    if missing_suffixes:
        raise RuntimeError(f"Missing language LoRA targets: {missing_suffixes}")
    if local_rank == 0:
        print("LoRA variant: 8B restricted loss, language attention only", flush=True)
        print(f"Resolved target modules: {len(language_targets)}", flush=True)
        model.print_trainable_parameters()
        print(f"Trainable parameter tensors: {len(trainable_names)}", flush=True)
        print(f"Model load: {(time.perf_counter() - load_started) / 60:.2f} min", flush=True)

    class RestrictedFourClassTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            supervised = labels.ne(-100)
            if not supervised.any(dim=1).all():
                raise RuntimeError("Every example must contain one supervised answer token.")
            answer_positions = supervised.to(torch.int64).argmax(dim=1)
            if (answer_positions == 0).any():
                raise RuntimeError("Answer token cannot occur at position zero.")
            batch_indices = torch.arange(labels.shape[0], device=labels.device)
            vocabulary_logits = outputs.logits[batch_indices, answer_positions - 1]
            label_token_ids = label_token_ids_cpu.to(vocabulary_logits.device)
            class_logits = vocabulary_logits.index_select(-1, label_token_ids).float()
            target_token_ids = labels[batch_indices, answer_positions]
            matches = target_token_ids[:, None].eq(label_token_ids[None, :])
            if not matches.any(dim=1).all():
                raise RuntimeError("A target token is outside the restricted four-label vocabulary.")
            class_targets = matches.to(torch.int64).argmax(dim=1)
            loss = F.cross_entropy(class_logits, class_targets)
            return (loss, outputs) if return_outputs else loss

    def restrict_logits_for_metrics(logits, labels):
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        supervised = labels.ne(-100)
        answer_positions = supervised.to(torch.int64).argmax(dim=1)
        batch_indices = torch.arange(labels.shape[0], device=labels.device)
        label_token_ids = label_token_ids_cpu.to(logits.device)
        return logits[batch_indices, answer_positions - 1].index_select(-1, label_token_ids)

    def compute_metrics(prediction):
        class_logits = np.asarray(prediction.predictions)
        labels = np.asarray(prediction.label_ids)
        target_token_ids = np.array(
            [row[np.flatnonzero(row != -100)[0]] for row in labels],
            dtype=np.int64,
        )
        token_to_class = {token_id: index for index, token_id in enumerate(collator.label_token_ids)}
        targets = np.array([token_to_class[int(token_id)] for token_id in target_token_ids])
        predictions = class_logits.argmax(axis=-1)
        metrics = {"macro_f1": f1_score(targets, predictions, average="macro")}
        per_class = f1_score(targets, predictions, labels=[0, 1, 2, 3], average=None, zero_division=0)
        metrics.update({f"f1_class_{index}": float(score) for index, score in enumerate(per_class)})
        return metrics

    class QuarterMilestoneCallback(TrainerCallback):
        """Evaluate and save at 25%, 50%, 75%, and 100% of optimizer steps."""

        def on_train_begin(self, args, state, control, **kwargs):
            self.milestones = {
                max(1, int(state.max_steps * fraction + 0.5))
                for fraction in (0.25, 0.50, 0.75, 1.00)
            }
            if state.is_world_process_zero:
                print(f"Evaluation/checkpoint milestones: {sorted(self.milestones)}", flush=True)
            return control

        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step in self.milestones:
                control.should_evaluate = True
                control.should_save = True
            return control

    train_dataset = ManifestDataset(args.train_manifest, random_swap=True)
    validation_dataset = ManifestDataset(args.validation_manifest, random_swap=False)
    if local_rank == 0:
        print(f"Train rows: {len(train_dataset)} | Validation rows: {len(validation_dataset)}", flush=True)

    training_args = TrainingArguments(
        output_dir=str(Path(args.work_root) / "trainer_output"),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=args.gradient_accumulation,
        num_train_epochs=args.epochs,
        learning_rate=5e-5,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        save_strategy="steps",
        # The callback below triggers the real quarter-run events. These large
        # equal values satisfy best-model strategy validation without adding events.
        eval_steps=10000,
        save_steps=10000,
        save_total_limit=4,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
        ddp_find_unused_parameters=False,
        seed=SEED,
    )
    trainer = RestrictedFourClassTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=collator,
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=restrict_logits_for_metrics,
        callbacks=[QuarterMilestoneCallback()],
    )

    torch.cuda.synchronize()
    train_started = time.perf_counter()
    result = trainer.train()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - train_started
    final_validation = trainer.evaluate()

    if trainer.is_world_process_zero():
        adapter_path = Path(args.adapter_dir)
        adapter_path.mkdir(parents=True, exist_ok=True)
        trainer.save_model(adapter_path)
        processor.save_pretrained(adapter_path)
        metrics = dict(result.metrics)
        metrics.update({f"best_{key}": value for key, value in final_validation.items()})
        metrics.update(
            {
                "wall_seconds": elapsed,
                "wall_minutes": elapsed / 60,
                "optimizer_steps": int(trainer.state.global_step),
                "seconds_per_optimizer_step": elapsed / max(1, trainer.state.global_step),
                "peak_gpu_gib_rank0": torch.cuda.max_memory_allocated() / 2**30,
                "best_checkpoint": trainer.state.best_model_checkpoint,
                "best_metric": trainer.state.best_metric,
                "train_rows": len(train_dataset),
                "validation_rows": len(validation_dataset),
                "loss_type": "restricted_four_class_cross_entropy",
            }
        )
        with (adapter_path / "training_metrics.json").open("w", encoding="utf-8") as handle:
            json.dump(metrics, handle, indent=2)
        print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


Writing train_restricted4_ddp.py


In [7]:
TRAIN_SCRIPT_PATH = Path('train_restricted4_ddp.py').resolve()
train_command = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--multi_gpu', '--num_processes', '2',
    str(TRAIN_SCRIPT_PATH),
    '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST),
    '--adapter-dir', str(ADAPTER_DIR),
    '--work-root', str(WORK_ROOT),
    '--model-path', MODEL_PATH,
    '--epochs', str(NUM_EPOCHS),
    '--gradient-accumulation', str(GRADIENT_ACCUMULATION),
]
launch_env = dict(os.environ, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')
print('Launching:', ' '.join(train_command), flush=True)
started = time.perf_counter()
subprocess.run(train_command, check=True, env=launch_env)
print(f'Training and four validations: {(time.perf_counter()-started)/60:.2f} min')
metrics_path = ADAPTER_DIR / 'training_metrics.json'
if metrics_path.exists():
    print(metrics_path.read_text())


Launching: /usr/bin/python3 -m accelerate.commands.launch --multi_gpu --num_processes 2 /kaggle/working/train_restricted4_ddp.py --train-manifest /kaggle/working/astroclimb_qwen3vl8b_restricted_validation/train_9200.jsonl --validation-manifest /kaggle/working/astroclimb_qwen3vl8b_restricted_validation/validation_800.jsonl --adapter-dir /kaggle/working/astroclimb_qwen3vl8b_restricted_validation/best_adapter --work-root /kaggle/working/astroclimb_qwen3vl8b_restricted_validation --model-path Qwen/Qwen3-VL-8B-Instruct --epochs 2 --gradient-accumulation 8


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.


Label token IDs: [15, 16, 17, 18]


Loading checkpoint shards: 100%|██████████| 4/4 [01:49<00:00, 27.30s/it]


LoRA variant: 8B restricted loss, language attention only
Resolved target modules: 144
trainable params: 15,335,424 || all params: 8,782,459,120 || trainable%: 0.1746
Trainable parameter tensors: 288
Model load: 3.11 min
Train rows: 9200 | Validation rows: 800
Evaluation/checkpoint milestones: [288, 575, 863, 1150]


...

{'train_runtime': 24658.043, 'train_samples_per_second': 0.746, 'train_steps_per_second': 0.047, 'train_loss': 0.8096995623215385, 'epoch': 2.0}


100%|██████████| 400/400 [06:25<00:00,  1.04it/s]


{
  "train_runtime": 24658.043,
  "train_samples_per_second": 0.746,
  "train_steps_per_second": 0.047,
  "total_flos": 4.009483822956544e+17,
  "train_loss": 0.8096995623215385,
  "epoch": 2.0,
  "best_eval_loss": 0.6387216448783875,
  "best_eval_macro_f1": 0.7291643805741228,
  "best_eval_f1_class_0": 0.9291338582677166,
  "best_eval_f1_class_1": 0.6698795180722892,
  "best_eval_f1_class_2": 0.5378973105134475,
  "best_eval_f1_class_3": 0.779746835443038,
  "best_eval_runtime": 386.8546,
  "best_eval_samples_per_second": 2.068,
  "best_eval_steps_per_second": 1.034,
  "best_epoch": 2.0,
  "wall_seconds": 24659.958692988,
  "wall_minutes": 410.9993115498,
  "optimizer_steps": 1150,
  "seconds_per_optimizer_step": 21.443442341728694,
  "peak_gpu_gib_rank0": 11.54600715637207,
  "best_checkpoint": "/kaggle/working/astroclimb_qwen3vl8b_restricted_validation/trainer_output/checkpoint-863",
  "best_metric": 0.7291643805741228,
  "train_rows": 9200,
  "validation_rows": 800,
  "loss_type": 

[rank0]:[W916 18:22:22.615979987 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Training and four validations: 421.29 min
{
  "train_runtime": 24658.043,
  "train_samples_per_second": 0.746,
  "train_steps_per_second": 0.047,
  "total_flos": 4.009483822956544e+17,
  "train_loss": 0.8096995623215385,
  "epoch": 2.0,
  "best_eval_loss": 0.6387216448783875,
  "best_eval_macro_f1": 0.7291643805741228,
  "best_eval_f1_class_0": 0.9291338582677166,
  "best_eval_f1_class_1": 0.6698795180722892,
  "best_eval_f1_class_2": 0.5378973105134475,
  "best_eval_f1_class_3": 0.779746835443038,
  "best_eval_runtime": 386.8546,
  "best_eval_samples_per_second": 2.068,
  "best_eval_steps_per_second": 1.034,
  "best_epoch": 2.0,
  "wall_seconds": 24659.958692988,
  "wall_minutes": 410.9993115498,
  "optimizer_steps": 1150,
  "seconds_per_optimizer_step": 21.443442341728694,
  "peak_gpu_gib_rank0": 11.54600715637207,
  "best_checkpoint": "/kaggle/working/astroclimb_qwen3vl8b_restricted_validation/trainer_output/checkpoint-863",
  "best_metric": 0.7291643805741228,
  "train_rows": 9200,

## Two-GPU inference on all 10,000 test rows

Each process loads the selected adapter on one T4 and predicts half of the test manifest. Keep `TEST_LIMIT=None` for a valid complete submission.


### Visible two-GPU inference worker

This cell writes the complete inference worker as normal Python source before launching it on both T4 GPUs.


In [8]:
%%writefile infer_ddp.py
import argparse
import csv
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
TARGET_COLUMNS = ["same_figure", "same_paper", "related_papers", "unrelated_papers"]
SYSTEM_PROMPT = """You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.""".strip()


def shorten_caption(text, max_chars=MAX_TEXT_CHARS):
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + "\n[...middle truncated...]\n" + text[-half:]


def object_content(number, obj):
    if obj["kind"] == "image":
        with Image.open(obj["value"]) as source:
            image = source.convert("RGB")
        return [
            {"type": "text", "text": f"Object {number} is a scientific figure:"},
            {"type": "image", "image": image},
        ]
    return [{"type": "text", "text": f"Object {number} is a figure caption:\n{shorten_caption(obj['value'])}"}]


def build_messages(row, swap=False):
    obj_1, obj_2 = row["obj_1"], row["obj_2"]
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({"type": "text", "text": "Classify their relationship. Reply with one digit only."})
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": content},
    ]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--test-manifest", required=True)
    parser.add_argument("--model-path", required=True)
    parser.add_argument("--adapter-dir", required=True)
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--test-limit", type=int, default=-1)
    parser.add_argument("--swap-tta", action="store_true")
    parser.add_argument("--modality-mask", action="store_true")
    args = parser.parse_args()

    # Select this rank's GPU before Transformers/torchao can probe and initialize CUDA.
    rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "2"))
    torch.cuda.set_device(rank)

    # Imports occur in fresh accelerate workers, never in a fork of a CUDA-initialized kernel.
    from peft import PeftModel
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

    with Path(args.test_manifest).open("r", encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle]
    if args.test_limit >= 0:
        rows = rows[: args.test_limit]
    rows = rows[rank::world_size]

    processor = AutoProcessor.from_pretrained(args.adapter_dir, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = Qwen3VLForConditionalGeneration.from_pretrained(
        args.model_path,
        quantization_config=quantization,
        dtype=torch.float16,
        attn_implementation="sdpa",
        device_map={"": rank},
    )
    model = PeftModel.from_pretrained(base, args.adapter_dir)
    model.eval()
    model.config.use_cache = True
    token_ids = []
    for digit in "0123":
        ids = processor.tokenizer.encode(digit, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f"Label {digit} is not one token: {ids}")
        token_ids.append(ids[0])

    @torch.inference_mode()
    def predict(row, swap=False):
        batch = processor.apply_chat_template(
            build_messages(row, swap=swap),
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        batch = {key: value.to(model.device) if torch.is_tensor(value) else value for key, value in batch.items()}
        logits = model(**batch).logits[0, -1, token_ids].float()
        if args.modality_mask and row["modality"] in {"CC", "II"}:
            logits[0] = float("-inf")
        return torch.softmax(logits, dim=-1).cpu().numpy()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"probabilities_rank{rank}.csv"
    started = time.perf_counter()
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", *[f"p_{name}" for name in TARGET_COLUMNS]])
        writer.writeheader()
        for index, row in enumerate(rows, start=1):
            probabilities = predict(row)
            if args.swap_tta:
                probabilities = 0.5 * (probabilities + predict(row, swap=True))
            writer.writerow(
                {"id": row["id"], **{f"p_{name}": float(probabilities[i]) for i, name in enumerate(TARGET_COLUMNS)}}
            )
            if index % 100 == 0:
                elapsed = time.perf_counter() - started
                print(
                    f"rank={rank} {index}/{len(rows)} {elapsed/index:.3f}s/row "
                    f"ETA={(elapsed/index)*(len(rows)-index)/3600:.2f}h",
                    flush=True,
                )
    elapsed = time.perf_counter() - started
    print(f"Rank {rank} finished {len(rows)} rows in {elapsed/3600:.2f}h", flush=True)


if __name__ == "__main__":
    main()


Writing infer_ddp.py


In [9]:
INFERENCE_SCRIPT_PATH = Path('infer_ddp.py').resolve()
if RUN_TEST_INFERENCE:
    if not INFERENCE_SCRIPT_PATH.is_file():
        raise FileNotFoundError(f'Inference worker was not written: {INFERENCE_SCRIPT_PATH}')
    inference_command = [
        sys.executable, '-m', 'accelerate.commands.launch',
        '--multi_gpu', '--num_processes', '2',
        str(INFERENCE_SCRIPT_PATH),
        '--test-manifest', str(TEST_MANIFEST),
        '--model-path', MODEL_PATH,
        '--adapter-dir', str(ADAPTER_DIR),
        '--output-dir', str(PREDICTION_DIR),
        '--test-limit', str(-1 if TEST_LIMIT is None else TEST_LIMIT),
    ]
    if USE_SWAP_TTA:
        inference_command.append('--swap-tta')
    if APPLY_MODALITY_MASK:
        inference_command.append('--modality-mask')
    print('Launching:', ' '.join(inference_command), flush=True)
    started = time.perf_counter()
    subprocess.run(inference_command, check=True, env=launch_env)
    print(f'Two-GPU inference: {(time.perf_counter()-started)/60:.2f} min')


## Merge probability shards and create `submission.csv`


In [10]:
if RUN_TEST_INFERENCE:
    shard_paths = [PREDICTION_DIR / f'probabilities_rank{rank}.csv' for rank in range(2)]
    for path in shard_paths:
        if not path.exists():
            raise FileNotFoundError(f'Missing inference shard: {path}')
    probabilities = pd.concat([pd.read_csv(path, dtype={'id': str}) for path in shard_paths], ignore_index=True)
    if probabilities['id'].duplicated().any():
        raise ValueError('Duplicate IDs found across inference shards.')
    with TEST_MANIFEST.open('r', encoding='utf-8') as handle:
        ordered_ids = [str(json.loads(line)['id']) for line in handle]
    if TEST_LIMIT is not None:
        ordered_ids = ordered_ids[:TEST_LIMIT]
    probabilities = probabilities.set_index('id').loc[ordered_ids].reset_index()
    probability_columns = [f'p_{name}' for name in TARGET_COLUMNS]
    if probabilities[probability_columns].isna().any().any():
        raise ValueError('Missing probabilities in merged output.')
    predicted_classes = probabilities[probability_columns].to_numpy().argmax(axis=1)
    submission = pd.DataFrame({'id': probabilities['id']})
    for class_index, name in enumerate(TARGET_COLUMNS):
        submission[name] = (predicted_classes == class_index).astype(int)
    submission.to_csv(SUBMISSION_PATH, index=False, lineterminator='\n')

    assert submission.columns.tolist() == ['id', *TARGET_COLUMNS]
    assert submission['id'].is_unique
    assert submission[TARGET_COLUMNS].isin([0, 1]).all().all()
    assert (submission[TARGET_COLUMNS].sum(axis=1) == 1).all()
    expected_rows = EXPECTED_TEST_ROWS if TEST_LIMIT is None else TEST_LIMIT
    assert len(submission) == expected_rows
    print('Submission:', SUBMISSION_PATH)
    print('Rows:', len(submission))
    print('Prediction counts:', submission[TARGET_COLUMNS].sum().to_dict())
    display(submission.head())


## Output artifacts

- Best adapter: `/kaggle/working/astroclimb_qwen3vl8b_restricted_validation/best_adapter/`
- Half-epoch checkpoints: `/kaggle/working/astroclimb_qwen3vl8b_restricted_validation/trainer_output/`
- Validation/training metrics: `best_adapter/training_metrics.json`
- Probability shards: `prediction_shards/probabilities_rank0.csv` and `probabilities_rank1.csv`
- Final submission: `/kaggle/working/astroclimb_qwen3vl8b_restricted_validation/submission.csv`
